# VN1 Ensemble Submission — Theta + AutoETS + SeasonalNaive + LightGBM

End-to-end pipeline for the [VN1 Forecasting Accuracy Challenge](https://www.datasource.ai/en/home/data-science-competitions-for-startups/phase-2-vn1-forecasting-accuracy-challenge).
Polars-first throughout; pandas appears only at the `statsforecast` library boundary.

**Strategy.**

1. Cross-validate four models on identical folds with the official competition metric.
2. Optimize a non-negative simplex of weights against out-of-sample CV predictions (no in-sample leakage).
3. Refit each model on the full training set with the same hyperparameters, blend with the CV-tuned weights, and apply a zero-variance naive override for SKUs whose tail is locked.
4. Pivot to the wide submission schema and write `submission.csv`.

**Why this works.** The competition metric `(|Σ err| + Σ|err|) / Σ y_true` penalizes both cumulative bias *and* dispersion, so it rewards model diversity (one model overshoots, another undershoots, the blend cancels). The four bases were chosen for complementary failure modes:

| Model | Captures | Misses |
|---|---|---|
| AutoTheta | trend + level decomposition | regime breaks |
| AutoETS | exponential-smoothed seasonality | sharp shocks |
| SeasonalNaive | last-year-same-week reference | drift |
| LightGBM | cross-series + lag interactions | very short series |


## 1 — Setup

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import numpy as np
import polars as pl

from vn1.data import load_full_data, process_wide_df, trim_leading_zeros
from vn1.metrics import comp_loss_mlforecast, comp_loss_polars, per_series_comp_loss
from vn1.validation import cross_validate, evaluate_predictions
from vn1.models import build_lgbm_recursive, fit_predict_lgbm, fit_predict_stats
from vn1.ensemble import apply_weights, optimize_weights
from vn1.submission import apply_zero_variance_override, write_submission
from vn1.style import apply_style

apply_style()
warnings.filterwarnings("ignore", message=r".*Found null values.*")
warnings.filterwarnings("ignore", message=r"The following series are too short.*")
warnings.filterwarnings("ignore", message=r"Could not find the number of physical cores.*")

## 2 — Configuration

Single source of truth for every knob the rest of the notebook reads. Pin `SEED` for reproducibility and bump `N_WINDOWS` if you want a more stable weight estimate (4 is enough for stable retail series; 6–8 reduces variance on intermittent ones).

In [ ]:
CONFIG = {
    "horizon":             13,         # competition horizon in weeks
    "n_windows":            4,         # rolling-origin folds for ensemble weight CV
    "step_size":           13,         # non-overlapping val windows
    "season_length":       52,         # weekly annual seasonality
    "freq_stats":      "W-MON",        # statsforecast freq (Monday-anchored weekly)
    "freq_ml":            "7d",        # mlforecast freq
    "lgbm_n_estimators":  500,         # fixed iter count in CV (use_cv=False) for honest folds
    "lgbm_use_cv_final": True,         # use LightGBMCV early-stopping for the final-fit iter count
    "lgbm_cv_n_windows":   4,
    "lgbm_cv_iters":  10_000,
    "submission_path":  Path("artifacts/submission.csv"),
    "seed":                42,
}

MODELS = ["Theta", "AutoETS", "SNaive", "LGBM"]
CONFIG

## 3 — Load + preprocess (polars-lazy)

`load_full_data` issues two HTTP reads (Phase 0 + Phase 1 sales CSVs from datasource.ai) and concatenates them horizontally. `process_wide_df` unpivots to the Nixtla `(unique_id, ds, y)` long schema; `trim_leading_zeros` drops pre-launch padding zeros that would otherwise distort the lag features.

In [ ]:
wide = load_full_data()
long = trim_leading_zeros(process_wide_df(wide))
sales = long.collect()  # materialize once; everything downstream re-lazies as needed

n_series = sales["unique_id"].n_unique()
n_weeks = sales["ds"].n_unique()
print(f"{n_series:>6,} unique series  ×  {n_weeks:>3} distinct weeks  =  {sales.height:>9,} rows")
print(f"Date range: {sales['ds'].min()} → {sales['ds'].max()}")

### Quick sanity check — zero-rate + history length distribution

Retail data at the SKU level is typically intermittent. If a large share of series is mostly zeros, the LightGBM model needs the zero-variance override to avoid hallucinating noise; the statistical models are mostly immune.

In [ ]:
diag = (
    sales.lazy()
    .group_by("unique_id")
    .agg(
        n_obs=pl.len(),
        zero_rate=(pl.col("y") == 0).mean(),
        max_y=pl.col("y").max(),
        std_y=pl.col("y").std(),
    )
    .collect()
)
print("history length quantiles:", diag["n_obs"].quantile(0.05), diag["n_obs"].quantile(0.5), diag["n_obs"].quantile(0.95))
print("zero-rate quantiles:     ", round(diag["zero_rate"].quantile(0.05), 3), round(diag["zero_rate"].quantile(0.5), 3), round(diag["zero_rate"].quantile(0.95), 3))
print("share intermittent (zero-rate ≥ 0.5):", round((diag["zero_rate"] >= 0.5).mean(), 3))

## 4 — Cross-validation: same folds, four models

`cross_validate(forecast_fn, sales, h, n_windows)` runs `forecast_fn(train, h)` against rolling-origin folds and returns long-format predictions joined to truth. We define one `forecast_fn` per model — the harness guarantees identical (train, val) splits across them, which is what makes the resulting weight optimization honest.

In [ ]:
def forecast_stats(train: pl.LazyFrame, h: int) -> pl.DataFrame:
    return fit_predict_stats(
        train,
        h=h,
        season_length=CONFIG["season_length"],
        freq=CONFIG["freq_stats"],
    )

def forecast_lgbm_cv(train: pl.LazyFrame, h: int) -> pl.DataFrame:
    # Fast path: fixed n_estimators per fold so per-fold scores are comparable.
    return fit_predict_lgbm(
        train,
        h=h,
        n_estimators=CONFIG["lgbm_n_estimators"],
        freq=CONFIG["freq_ml"],
        season_length=CONFIG["season_length"],
    )

In [ ]:
%%time
cv_stats = cross_validate(
    forecast_stats, sales,
    h=CONFIG["horizon"],
    n_windows=CONFIG["n_windows"],
    step_size=CONFIG["step_size"],
)
print("cv_stats:", cv_stats.shape, cv_stats.columns)

In [ ]:
%%time
cv_lgbm = cross_validate(
    forecast_lgbm_cv, sales,
    h=CONFIG["horizon"],
    n_windows=CONFIG["n_windows"],
    step_size=CONFIG["step_size"],
)
print("cv_lgbm:", cv_lgbm.shape, cv_lgbm.columns)

### Merge into a single CV frame

The harness emits one frame per model (each with its own model column). Joining on `(unique_id, ds, fold)` gives a single frame with all four model columns plus truth — the canonical input to `optimize_weights`.

In [ ]:
cv = cv_stats.join(
    cv_lgbm.select("unique_id", "ds", "fold", "LGBM"),
    on=["unique_id", "ds", "fold"],
    how="inner",
)
print("merged cv frame:", cv.shape)
cv.head()

## 5 — Per-model, per-fold scores

Before optimizing weights, sanity-check that every model is in the same ballpark and that no single fold dominates. If overall ≈ per-fold, the CV is honest. If overall ≪ per-fold, you have leakage. If one fold is wildly different, the metric is being driven by a small number of high-volume series and you should bootstrap.

In [ ]:
scores = evaluate_predictions(cv, pred_cols=MODELS)
scores.pivot(on="fold", index="model", values="comp_loss").sort("model")

## 6 — Optimize ensemble weights

Non-negative simplex (`w ≥ 0`, `Σw = 1`) minimizing `comp_loss` on the merged CV predictions. SLSQP with explicit constraints; multi-start (4 Dirichlet draws + equal-weights seed) for robustness against the metric's `|·|` non-smoothness.

In [ ]:
weights = optimize_weights(cv, pred_cols=MODELS, seed=CONFIG["seed"])
print("\noptimized weights:")
for m in MODELS:
    print(f"  {m:<10s}  {weights[m]:.4f}")
print(f"  sum       {sum(weights.values()):.6f}")

In [ ]:
blended_cv = apply_weights(cv, weights, out_col="Ensemble").collect()
ens_loss = comp_loss_polars(blended_cv, pred_col="Ensemble")
print(f"\nensemble CV comp_loss: {ens_loss:.4f}")
best_single = min(comp_loss_polars(cv, pred_col=m) for m in MODELS)
improvement = (best_single - ens_loss) / best_single
print(f"best single model:     {best_single:.4f}")
print(f"relative improvement:  {improvement:>+6.2%}")
assert ens_loss <= best_single + 1e-6, "ensemble worse than best single — investigate before submitting"

## 7 — Refit on full data + forecast

Now that the weights are locked, refit each model on the *entire* training history and forecast the next 13 weeks. For LightGBM we optionally use `LightGBMCV` early-stopping to set `num_iterations` honestly (slower but stronger on the leaderboard); the statistical bundle has no hyperparameters to tune so it's a single `fit_predict_stats` call.

In [ ]:
%%time
stats_full = fit_predict_stats(
    sales,
    h=CONFIG["horizon"],
    season_length=CONFIG["season_length"],
    freq=CONFIG["freq_stats"],
)
stats_full.head()

In [ ]:
%%time
lgbm_full = fit_predict_lgbm(
    sales,
    h=CONFIG["horizon"],
    use_cv=CONFIG["lgbm_use_cv_final"],
    cv_n_windows=CONFIG["lgbm_cv_n_windows"],
    cv_num_iterations=CONFIG["lgbm_cv_iters"],
    cv_metric=comp_loss_mlforecast,
    freq=CONFIG["freq_ml"],
    season_length=CONFIG["season_length"],
    n_estimators=CONFIG["lgbm_n_estimators"],
)
lgbm_full.head()

## 8 — Blend + zero-variance override + submit

Join all four model columns on `(unique_id, ds)`, compute the weighted ensemble, then apply the zero-variance naive override (locked-tail SKUs get last-value carried forward instead of LGBM's lag-feature noise). Finally pivot to the wide submission schema and write.

In [ ]:
full_preds = stats_full.join(lgbm_full, on=["unique_id", "ds"], how="inner")
blended = apply_weights(full_preds, weights, out_col="y_hat").collect()
finalized = apply_zero_variance_override(blended, sales, n_tail=2).collect()
print("finalized predictions:", finalized.shape)
finalized.head()

In [ ]:
out_path = write_submission(
    finalized,
    output_path=CONFIG["submission_path"],
)
print(f"wrote {out_path}")
print(f"  rows = {sum(1 for _ in open(out_path)) - 1:,}  (one per series)")
print(f"  cols = 3 + {CONFIG['horizon']}  (Client/Warehouse/Product + h dates)")

## 9 — Diagnostic: where the ensemble fails

Per-series competition loss on the most recent CV fold. The tail of this distribution is your roadmap: those are the SKUs the current ensemble doesn't capture, and they're where a per-segment model (e.g. a Croston-style intermittent forecaster, or a long-history-only LightGBM with cross-series features) would move the needle most.

In [ ]:
last_fold = blended_cv.filter(pl.col("fold") == blended_cv["fold"].max())
ps = per_series_comp_loss(last_fold, pred_col="Ensemble")
print("per-series ensemble loss — worst 10:")
ps.head(10)